# Tech Job Trends — Stage 2: Semantic Search + RAG Career Advisor

This notebook loads the ChromaDB collection created in Stage 1, retrieves the most relevant job-posting chunks, builds a grounded prompt, and asks Gemini to answer strictly from the retrieved context.


## Configuration

Set `GEMINI_API_KEY` in the environment before running.

For Google Colab, you can set it in the runtime environment (or use Colab Secrets) rather than hard-coding it in the notebook.

`CHROMA_DB_PATH` must point to the ChromaDB directory produced by Stage 1.


In [ ]:
import os
from pathlib import Path

import chromadb
from chromadb.utils import embedding_functions
from google import genai
from google.genai import types

CHROMA_DB_PATH = Path("./chroma_db")
COLLECTION_NAME = "tech_jobs"
TOP_K = 3
MAX_DISTANCE = 0.8

# Use text-generation models only. Keep the list short and easy to maintain.
CANDIDATE_MODELS = [
    "models/gemini-3.5-flash",
    "models/gemini-2.5-flash",
    "models/gemini-2.5-pro",
    "models/gemini-flash-latest",
]


In [ ]:
class ChromaJobRetriever:
    """Semantic retrieval over the persistent ChromaDB collection."""

    def __init__(self, db_path: Path, collection_name: str):
        if not db_path.exists():
            raise FileNotFoundError(
                f"ChromaDB not found: {db_path}. Run Stage 1 first or update CHROMA_DB_PATH."
            )

        self.client = chromadb.PersistentClient(path=str(db_path))
        self.embedding_fn = embedding_functions.DefaultEmbeddingFunction()
        self.collection = self.client.get_collection(
            name=collection_name,
            embedding_function=self.embedding_fn,
        )

    def search(self, query: str, top_k: int = TOP_K, company_filter: str | None = None):
        where_clause = {"company": company_filter} if company_filter else None
        return self.collection.query(
            query_texts=[query],
            n_results=top_k,
            where=where_clause,
        )

    @staticmethod
    def print_results(results) -> None:
        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]

        print(f"\nFound {len(documents)} candidate chunks\n" + "=" * 60)
        for idx, (doc, meta, dist) in enumerate(
            zip(documents, metadatas, distances), start=1
        ):
            print(f"\nResult #{idx} | Cosine distance: {dist:.4f}")
            print(f"Company: {meta.get('company')}")
            print(f"Job title: {meta.get('job_title')}")
            print(f"Parent ID: {meta.get('parent_id')}")
            print(f"Text: {doc}")
            print("-" * 60)


In [ ]:
class RAGJobGenerator:
    """Generate a grounded career-advisor response from retrieved job context."""

    def __init__(self, api_key: str, candidate_models: list[str] | None = None):
        if not api_key:
            raise ValueError("GEMINI_API_KEY is missing. Set it in the environment.")
        self.client = genai.Client(api_key=api_key)
        self.candidate_models = candidate_models or CANDIDATE_MODELS

    def build_prompt(self, query: str, retrieved_results) -> str:
        documents = retrieved_results["documents"][0]
        metadatas = retrieved_results["metadatas"][0]
        distances = retrieved_results["distances"][0]

        context_blocks = []
        for doc, meta, dist in zip(documents, metadatas, distances):
            if dist > MAX_DISTANCE:
                continue

            context_blocks.append(
                f"- Job title: {meta.get('job_title')}\n"
                f"  Company: {meta.get('company')}\n"
                f"  Details: {doc}"
            )

        context_text = (
            "\n\n".join(context_blocks)
            if context_blocks
            else "No sufficiently relevant jobs were found in the current database."
        )

        return f"""
You are an expert AI Career Advisor analyzing tech job market data.
Answer the user's question STRICTLY based on the provided Job Context below.
Do not invent jobs, companies, requirements, locations, or other facts.
If the context does not contain relevant information, say:
"I couldn't find relevant jobs matching your criteria in the current database."

### Job Context
{context_text}

### User Query
{query}

### Answer
"""

    def generate_answer(self, prompt: str) -> str:
        config = types.GenerateContentConfig(temperature=0.1)

        last_error = None
        for model in self.candidate_models:
            try:
                print(f"Trying model: {model}")
                response = self.client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=config,
                )
                if response.text:
                    return response.text
            except Exception as exc:
                last_error = exc
                print(f"Model unavailable: {model}")

        raise RuntimeError("All configured Gemini models failed.") from last_error


In [ ]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
USER_QUERY = "Python developer with machine learning experience"

retriever = ChromaJobRetriever(
    db_path=CHROMA_DB_PATH,
    collection_name=COLLECTION_NAME,
)

search_results = retriever.search(USER_QUERY, top_k=TOP_K)
retriever.print_results(search_results)

generator = RAGJobGenerator(api_key=GEMINI_API_KEY)
prompt = generator.build_prompt(USER_QUERY, search_results)
ai_response = generator.generate_answer(prompt)

print("\n" + "=" * 60)
print("Final AI Response")
print("=" * 60)
print(ai_response)


## Example workflow

1. Run `01_build_chroma_index.ipynb` to create/update the vector index.
2. Set `GEMINI_API_KEY`.
3. Point `CHROMA_DB_PATH` at the generated index.
4. Change `USER_QUERY` to test other job-search questions.
